# KDIC BM25 기본 검색 평가기

형태소 분석 없이 기본 정규식 토큰화와 BM25 점수로 427개 청크를 검색하고, 평가 질문 113개의 Top-10 결과를 Gold 청크와 비교합니다.

- 외부 API와 API 키를 사용하지 않습니다.
- GPU도 필요하지 않습니다.
- Dense·BGE-M3 Sparse와 동일한 업무 필터·Gold·Top-K·8개 지표를 사용합니다.
- 이번 결과는 다음 실험인 BM25-Nori와 비교할 기본 기준선입니다.

## 준비 파일

1. `KDIC_BM25_기본_평가기.zip`
2. `KDIC_output.zip`
3. `평가데이터셋_검색평가지표용.xlsx`

In [ ]:
%pip -q install -U "pandas==2.2.2" "openpyxl>=3.1,<4"

import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_ROOT=Path('/content/kdic_bm25_evaluation')
EVALUATOR_ROOT=WORK_ROOT/'evaluator'
RESULT_ROOT=WORK_ROOT/'results'
WORK_ROOT.mkdir(parents=True,exist_ok=True)
print('환경 준비 완료:',WORK_ROOT)
print('BM25 기본 평가는 API 키와 GPU를 사용하지 않습니다.')

## 1. 파일 업로드

아래 셀을 실행하고 준비한 파일 3개를 한 번에 선택합니다.

In [ ]:
from google.colab import files
uploaded=files.upload()
for filename,content in uploaded.items():
    target=WORK_ROOT/filename
    target.write_bytes(content)
    print(f'업로드: {target.name} ({target.stat().st_size:,} bytes)')

In [ ]:
dataset_candidates=list(WORK_ROOT.glob('*.xlsx'))
kdic_candidates=[p for p in WORK_ROOT.glob('*.zip') if p.name=='KDIC_output.zip' or p.name.startswith('KDIC_output (')]
evaluator_candidates=[p for p in WORK_ROOT.glob('*.zip') if 'BM25' in p.name and '평가기' in p.name and 'Nori' not in p.name and '평가결과' not in p.name]
if not dataset_candidates: raise RuntimeError('평가 XLSX를 찾을 수 없습니다.')
if not kdic_candidates: raise RuntimeError('KDIC_output.zip을 찾을 수 없습니다.')
if not evaluator_candidates: raise RuntimeError('KDIC_BM25_기본_평가기.zip을 찾을 수 없습니다.')
def choose_latest(items): return max(items,key=lambda p:p.stat().st_mtime)
DATASET_PATH=choose_latest(dataset_candidates); KDIC_ZIP_PATH=choose_latest(kdic_candidates); EVALUATOR_ZIP_PATH=choose_latest(evaluator_candidates)
if EVALUATOR_ROOT.exists(): shutil.rmtree(EVALUATOR_ROOT)
EVALUATOR_ROOT.mkdir(parents=True)
with zipfile.ZipFile(EVALUATOR_ZIP_PATH) as archive: archive.extractall(EVALUATOR_ROOT)
scripts=list(EVALUATOR_ROOT.rglob('evaluate_bm25_basic.py'))
if len(scripts)!=1: raise RuntimeError(f'평가 스크립트를 하나로 결정할 수 없습니다: {scripts}')
EVALUATOR_SCRIPT=scripts[0]
print('평가데이터셋:',DATASET_PATH.name); print('검색 데이터:',KDIC_ZIP_PATH.name); print('평가기:',EVALUATOR_ZIP_PATH.name)

## 2. 비교 조건 설정

처음 비교에서는 기본값을 유지하세요. 파라미터 튜닝은 기본 BM25·Nori·Dense·Sparse 비교가 끝난 뒤 별도 실험으로 진행하는 것이 좋습니다.

In [ ]:
K1=1.5
B=0.75
TOP_K=10
print({'검색 방식':'BM25 Basic','토큰화':'정규식 기반, 형태소 분석 없음','k1':K1,'b':B,'Top-K':TOP_K,'업무 필터':'Gold 업무 사전 필터'})

## 3. Dry-run

검색 실행 전에 질문·청크·Gold 연결을 검사합니다.

In [ ]:
dry_dir=WORK_ROOT/'results_dry'
cmd=[sys.executable,str(EVALUATOR_SCRIPT),'--dataset',str(DATASET_PATH),'--kdic-zip',str(KDIC_ZIP_PATH),'--output-dir',str(dry_dir),'--top-k',str(TOP_K),'--k1',str(K1),'--b',str(B),'--dry-run']
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)

## 4. BM25 기본 전체 평가

API나 모델 다운로드가 없어 일반적으로 짧은 시간 안에 완료됩니다.

In [ ]:
RESULT_ROOT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,str(EVALUATOR_SCRIPT),'--dataset',str(DATASET_PATH),'--kdic-zip',str(KDIC_ZIP_PATH),'--output-dir',str(RESULT_ROOT),'--top-k',str(TOP_K),'--k1',str(K1),'--b',str(B)]
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)

## 5. 평가 결과 확인

In [ ]:
summary=json.loads((RESULT_ROOT/'summary.json').read_text(encoding='utf-8'))
overall=pd.DataFrame([summary['overall']]); by_domain=pd.read_csv(RESULT_ROOT/'summary_by_domain.csv'); questions=pd.read_csv(RESULT_ROOT/'question_results.csv')
print('검색 방식:',summary['retriever']); print('전체 평가 결과'); display(overall); print('도메인별 평가 결과'); display(by_domain); print('Hit@3 실패 질문 예시')
display(questions.loc[questions['hit_at_3']==0,['evaluation_id','question','domain','gold_chunk_ids','retrieved_chunk_ids']].head(10))

## 6. 결과 다운로드

다운로드한 ZIP을 KDIC 검색 품질 비교 대시보드에 넣으면 Dense·Sparse 결과와 함께 비교할 수 있습니다.

In [ ]:
from google.colab import files
archive_path=shutil.make_archive('/content/KDIC_BM25_기본_평가결과','zip',RESULT_ROOT)
print('결과 압축:',archive_path)
files.download(archive_path)